# Generate IGNODE test model artifacts

**Save a copy to your Drive first** (File → Save a copy in Drive) so your edits persist.

This notebook builds tiny test artifacts in every model format the IGNODE Custom Model Upload wizard accepts. The models have random or minimally-trained weights — they exist to exercise the upload → deploy → predict pipeline, not for accuracy.

Each cell below generates one artifact and downloads it as a ready-to-upload ZIP. Drag any ZIP into IGNODE → Custom Model Upload to test the platform end-to-end.

**Coverage:**

| Format      | Classification     | Regression          | Image classification     |
|-------------|--------------------|---------------------|--------------------------|
| `lgbm_text` | iris               | diabetes            | — (not supported)        |
| `xgb_json`  | iris               | diabetes            | — (not supported)        |
| `onnx`      | iris (sklearn)     | diabetes (sklearn)  | random-weight MobileNetV2 |
| `tflite`    | — (not supported)  | — (not supported)   | tiny CNN                 |

Last verified: May 30, 2026

In [ ]:
# Install the libraries this notebook needs.
#
# torch + torchvision come pre-installed on Colab; tensorflow ships too.
# The ML libraries are pinned for reproducibility; the onnx ecosystem packages
# are unpinned because they co-evolve and exact pins break across releases.
!pip install lightgbm==4.5.0 xgboost==2.1.2 scikit-learn==1.5.2 \
             onnx onnxmltools skl2onnx onnxconverter-common

import sys
import lightgbm as lgb
import xgboost as xgb
import sklearn
import onnx
import skl2onnx
import torch
import torchvision
import tensorflow as tf

print(f'Python:       {sys.version.split()[0]}')
print(f'LightGBM:     {lgb.__version__}')
print(f'XGBoost:      {xgb.__version__}')
print(f'scikit-learn: {sklearn.__version__}')
print(f'onnx:         {onnx.__version__}')
print(f'torch:        {torch.__version__}')
print(f'tensorflow:   {tf.__version__}')

In [ ]:
# Shared helpers — sidecar writers + ZIP bundler + auto-download.
#
# All sidecars follow the canonical IR-2.15 schema:
#   class_labels.json    → {"schema_version": 1, "class_labels": [...]}
#   feature_columns.json → {"schema_version": 1, "feature_columns": [...], "label_columns": [...]}
#   preprocess_config.json → image preprocessing (ImageNet defaults)
import json, os, zipfile, shutil
from datetime import datetime
from google.colab import files

ROOT = '/content/test-artifacts'
os.makedirs(ROOT, exist_ok=True)

def prepare(kind):
    folder = os.path.join(ROOT, kind)
    if os.path.isdir(folder):
        shutil.rmtree(folder)
    os.makedirs(folder)
    return folder

def write_class_labels(folder, labels):
    with open(os.path.join(folder, 'class_labels.json'), 'w') as f:
        json.dump({'schema_version': 1, 'class_labels': list(labels)}, f, indent=2)

def write_feature_columns(folder, feature_columns, label_columns=None):
    with open(os.path.join(folder, 'feature_columns.json'), 'w') as f:
        json.dump({
            'schema_version': 1,
            'feature_columns': list(feature_columns),
            'label_columns': list(label_columns or []),
        }, f, indent=2)

def write_preprocess_config(folder):
    with open(os.path.join(folder, 'preprocess_config.json'), 'w') as f:
        json.dump({
            'input_size': [224, 224],
            'mean': [0.485, 0.456, 0.406],
            'std':  [0.229, 0.224, 0.225],
            'channel_order': 'RGB',
            'image_format': 'CHW',
            'rescale': 'imagenet',
            'resize_method': 'center_crop',
            'resize_short_side': 256,
        }, f, indent=2)

def bundle_and_download(folder, kind):
    ts = datetime.now().strftime('%Y%m%d-%H%M')
    zip_name = f'ignode-test-{kind}-{ts}.zip'
    zip_path = os.path.join(ROOT, zip_name)
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for fname in sorted(os.listdir(folder)):
            full = os.path.join(folder, fname)
            if os.path.isfile(full):
                zf.write(full, arcname=fname)
    print(f'Bundled {kind} → {zip_name} ({os.path.getsize(zip_path)/1024:.1f} KB)')
    files.download(zip_path)

print('helpers ready')

## LightGBM

In [ ]:
from sklearn.datasets import load_iris
iris = load_iris()
X, y = iris.data, iris.target

model = lgb.LGBMClassifier(n_estimators=20, learning_rate=0.1, verbose=-1)
model.fit(X, y)

folder = prepare('lightgbm-classifier')
model.booster_.save_model(os.path.join(folder, 'model.txt'))
write_class_labels(folder, iris.target_names)
write_feature_columns(folder, iris.feature_names, label_columns=['species'])
bundle_and_download(folder, 'lightgbm-classifier')

In [ ]:
from sklearn.datasets import load_diabetes
diabetes = load_diabetes()
X, y = diabetes.data, diabetes.target

model = lgb.LGBMRegressor(n_estimators=20, learning_rate=0.1, verbose=-1)
model.fit(X, y)

folder = prepare('lightgbm-regressor')
model.booster_.save_model(os.path.join(folder, 'model.txt'))
write_feature_columns(folder, diabetes.feature_names, label_columns=['progression'])
bundle_and_download(folder, 'lightgbm-regressor')

## XGBoost

Uses `get_booster().save_model()` rather than the sklearn wrapper's `save_model()`— newer xgboost versions gate the wrapper call on sklearn metadata that's brittle across releases, and the IGNODE xgb_json loader reads the raw booster JSON anyway.

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target

model = xgb.XGBClassifier(n_estimators=20, learning_rate=0.1, eval_metric='mlogloss')
model.fit(X, y)

folder = prepare('xgboost-classifier')
model.get_booster().save_model(os.path.join(folder, 'model.json'))
write_class_labels(folder, iris.target_names)
write_feature_columns(folder, iris.feature_names, label_columns=['species'])
bundle_and_download(folder, 'xgboost-classifier')

In [ ]:
diabetes = load_diabetes()
X, y = diabetes.data, diabetes.target

model = xgb.XGBRegressor(n_estimators=20, learning_rate=0.1)
model.fit(X, y)

folder = prepare('xgboost-regressor')
model.get_booster().save_model(os.path.join(folder, 'model.json'))
write_feature_columns(folder, diabetes.feature_names, label_columns=['progression'])
bundle_and_download(folder, 'xgboost-regressor')

## Sklearn → ONNX (tabular)

Random-forest classifier + regressor exported via `skl2onnx`. Uses opset 15 — newer opsets aren't supported by `skl2onnx` for tree models yet.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

iris = load_iris()
X, y = iris.data, iris.target

model = RandomForestClassifier(n_estimators=10, random_state=0)
model.fit(X, y)

initial_types = [('input', FloatTensorType([None, len(iris.feature_names)]))]
onnx_model = convert_sklearn(model, initial_types=initial_types, target_opset=15)

folder = prepare('sklearn-onnx-classifier')
onnx.save_model(onnx_model, os.path.join(folder, 'model.onnx'))
write_class_labels(folder, iris.target_names)
write_feature_columns(folder, iris.feature_names, label_columns=['species'])
bundle_and_download(folder, 'sklearn-onnx-classifier')

In [ ]:
from sklearn.ensemble import RandomForestRegressor

diabetes = load_diabetes()
X, y = diabetes.data, diabetes.target

model = RandomForestRegressor(n_estimators=10, random_state=0)
model.fit(X, y)

initial_types = [('input', FloatTensorType([None, len(diabetes.feature_names)]))]
onnx_model = convert_sklearn(model, initial_types=initial_types, target_opset=15)

folder = prepare('sklearn-onnx-regressor')
onnx.save_model(onnx_model, os.path.join(folder, 'model.onnx'))
write_feature_columns(folder, diabetes.feature_names, label_columns=['progression'])
bundle_and_download(folder, 'sklearn-onnx-regressor')

## PyTorch → ONNX (image classification)

MobileNetV2 architecture with **random weights** — fast to export, small enough as a fixture. Predictions are meaningless until the customer retrains; we only need a valid ONNX image-classifier file.

In [ ]:
from torchvision import models

model = models.mobilenet_v2(weights=None)
model.eval()

folder = prepare('onnx-image-classifier')
dummy = torch.randn(1, 3, 224, 224)
torch.onnx.export(
    model, dummy, os.path.join(folder, 'model.onnx'),
    input_names=['input'], output_names=['Score'],
    opset_version=18,
    dynamic_axes={'input': {0: 'batch_size'}, 'Score': {0: 'batch_size'}},
    do_constant_folding=True,
)
write_class_labels(folder, [f'class_{i}' for i in range(1000)])
write_preprocess_config(folder)
bundle_and_download(folder, 'onnx-image-classifier')

## TFLite (image classification)

Tiny CNN built with `tf.keras` then converted via `tf.lite.TFLiteConverter`. Random weights — same reasoning as the ONNX image case above.

In [ ]:
inputs = tf.keras.Input(shape=(224, 224, 3), name='input')
x = tf.keras.layers.Conv2D(8, 3, activation='relu')(inputs)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
outputs = tf.keras.layers.Dense(10, activation='softmax', name='Score')(x)
model = tf.keras.Model(inputs, outputs)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_bytes = converter.convert()

folder = prepare('tflite-image-classifier')
with open(os.path.join(folder, 'model.tflite'), 'wb') as f:
    f.write(tflite_bytes)
write_class_labels(folder, [f'class_{i}' for i in range(10)])
write_preprocess_config(folder)
bundle_and_download(folder, 'tflite-image-classifier')

## Next steps

Each downloaded ZIP is wizard-ready. Drag it into:

> IGNODE → ML Factory → Add Model → Custom Model Upload

The wizard auto-detects the format (magic bytes), pre-fills metadata, and lets you deploy to an ML Inference (UINF) server.